# scOPE end-to-end example

This notebook walks through the complete two-phase scOPE workflow:
1. **Phase 1** — learn latent space from bulk RNA-seq and train mutation classifiers.
2. **Phase 2** — project scRNA-seq into the bulk latent space, infer per-cell mutation probabilities.


In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt

from scope import BulkPipeline, SingleCellPipeline
from scope.io import load_mutation_labels
from scope.visualization import (
    compute_umap,
    plot_mutation_probabilities,
    plot_scree,
    plot_mutation_heatmap,
)
from scope.evaluation import evaluate_all


## 1. Load data

In [ ]:
# Replace with your actual paths
adata_bulk = ad.read_h5ad('data/bulk_cohort.h5ad')
mutation_labels = load_mutation_labels('data/mutations.csv', sample_col='sample_id')
adata_sc = ad.read_h5ad('data/sc_tumor.h5ad')

print(f'Bulk: {adata_bulk.n_obs} samples × {adata_bulk.n_vars} genes')
print(f'SC:   {adata_sc.n_obs} cells × {adata_sc.n_vars} genes')
mutation_labels.head()


## 2. Phase 1 — Bulk pipeline

In [ ]:
bulk_pipe = BulkPipeline(
    norm_method='cpm',        # CPM normalisation
    log1p=True,
    center=True,
    scale=True,
    decomposition='svd',      # truncated SVD
    n_components=50,
    classifier='logistic',    # regularised logistic regression
    classifier_kwargs={'C': 0.1, 'class_weight': 'balanced'},
)

bulk_pipe.fit(adata_bulk, mutation_labels, cv=5)
print('CV results:')
if bulk_pipe.cv_results_ is not None:
    print(bulk_pipe.cv_results_[['mutation', 'mean', 'std']].to_string())


In [ ]:
# Scree plot
scree = bulk_pipe.decomposer_.scree_data()
fig, ax = plot_scree(scree, max_components=30)
plt.show()


## 3. Phase 2 — Single-cell projection

In [ ]:
# Preprocessed bulk reference for moment matching
adata_bulk_pp = bulk_pipe.preprocessor_.transform(adata_bulk)

sc_pipe = SingleCellPipeline(
    bulk_pipeline=bulk_pipe,
    alignment_method='z_score_bulk',
    sc_min_counts=200,
    sc_min_genes=200,
)
sc_pipe.fit(adata_bulk_pp, adata_sc)
adata_sc = sc_pipe.transform(adata_sc)

# Show inferred mutation probability columns
prob_cols = [c for c in adata_sc.obs.columns if c.startswith('mutation_prob_')]
print('Mutation probability columns:', prob_cols)
adata_sc.obs[prob_cols].describe()


## 4. UMAP visualisation

In [ ]:
adata_sc = compute_umap(adata_sc, obsm_key=bulk_pipe.obsm_key_, n_neighbors=15)

fig = plot_mutation_probabilities(
    adata_sc,
    obsm_key='X_umap',
    ncols=3,
)
fig.savefig('mutation_probability_umap.pdf', bbox_inches='tight')
plt.show()


## 5. Cluster-level summary

In [ ]:
# Assumes leiden clustering was run e.g. via scanpy
# import scanpy as sc
# sc.pp.neighbors(adata_sc, use_rep=bulk_pipe.obsm_key_)
# sc.tl.leiden(adata_sc)

if 'leiden' in adata_sc.obs.columns:
    fig, ax = plot_mutation_heatmap(adata_sc, cluster_key='leiden')
    plt.show()


## 6. Save model

In [ ]:
bulk_pipe.save('models/bulk_pipeline.pkl')
adata_sc.write_h5ad('results/sc_with_mutation_probs.h5ad')
